In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

In [2]:
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_RAW:", DATA_RAW)

PROJECT_ROOT: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand
DATA_RAW: c:\temp\python_learning\ml_projects\ml_projects_batch_01\05_bike_sharing_demand\data\raw


In [3]:
train = pd.read_csv(DATA_RAW / "train.csv")

print("train shape:", train.shape)
display(train.head())

train shape: (10886, 12)


,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1


In [4]:
train["datetime"] = pd.to_datetime(train["datetime"])

train["year"] = train["datetime"].dt.year
train["month"] = train["datetime"].dt.month
train["day"] = train["datetime"].dt.day
train["hour"] = train["datetime"].dt.hour
train["weekday"] = train["datetime"].dt.weekday
train["is_weekend"] = train["weekday"].isin([5, 6]).astype(int)

display(train.head())

,datetime,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,casual,registered,count,year,month,day,hour,weekday,is_weekend
0,2011-01-01 00:00:00,1,0,0,1,9.84,14.395,81,0.0,3,13,16,2011,1,1,0,5,1
1,2011-01-01 01:00:00,1,0,0,1,9.02,13.635,80,0.0,8,32,40,2011,1,1,1,5,1
2,2011-01-01 02:00:00,1,0,0,1,9.02,13.635,80,0.0,5,27,32,2011,1,1,2,5,1
3,2011-01-01 03:00:00,1,0,0,1,9.84,14.395,75,0.0,3,10,13,2011,1,1,3,5,1
4,2011-01-01 04:00:00,1,0,0,1,9.84,14.395,75,0.0,0,1,1,2011,1,1,4,5,1


In [5]:
target_col = "count"

leakage_cols = ["casual", "registered"]

excluded_cols = [
    target_col,
    "casual",
    "registered",
    "datetime",
    "day",  # excluded from main comparison because of transfer-risk
]

feature_cols = [col for col in train.columns if col not in excluded_cols]

print("Target:", target_col)
print("Leakage columns:", leakage_cols)
print("Excluded columns:", excluded_cols)

print("\nFeature columns:")
print(feature_cols)
print("\nNumber of features:", len(feature_cols))

Target: count
Leakage columns: ['casual', 'registered']
Excluded columns: ['count', 'casual', 'registered', 'datetime', 'day']

Feature columns:
['season', 'holiday', 'workingday', 'weather', 'temp', 'atemp', 'humidity', 'windspeed', 'year', 'month', 'hour', 'weekday', 'is_weekend']

Number of features: 13


In [6]:
X = train[feature_cols].copy()
y = train[target_col].copy()

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

X shape: (10886, 13)
y shape: (10886,)


,season,holiday,workingday,weather,temp,atemp,humidity,windspeed,year,month,hour,weekday,is_weekend
0,1,0,0,1,9.84,14.395,81,0.0,2011,1,0,5,1
1,1,0,0,1,9.02,13.635,80,0.0,2011,1,1,5,1
2,1,0,0,1,9.02,13.635,80,0.0,2011,1,2,5,1
3,1,0,0,1,9.84,14.395,75,0.0,2011,1,3,5,1
4,1,0,0,1,9.84,14.395,75,0.0,2011,1,4,5,1


0    16
1    40
2    32
3    13
4     1
Name: count, dtype: int64

In [7]:
print("Leakage columns in X:")
print(set(X.columns).intersection(leakage_cols))

print("\nTarget in X:")
print(target_col in X.columns)

print("\nRaw datetime in X:")
print("datetime" in X.columns)

print("\nDay in main comparison X:")
print("day" in X.columns)

Leakage columns in X:
set()

Target in X:
False

Raw datetime in X:
False

Day in main comparison X:
False


In [8]:
local_train_mask = train["day"] <= 15
local_valid_mask = train["day"].between(16, 19)

X_train = X.loc[local_train_mask].copy()
y_train = y.loc[local_train_mask].copy()

X_valid = X.loc[local_valid_mask].copy()
y_valid = y.loc[local_valid_mask].copy()

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)
print("X_valid shape:", X_valid.shape)
print("y_valid shape:", y_valid.shape)

X_train shape: (8600, 13)
y_train shape: (8600,)
X_valid shape: (2286, 13)
y_valid shape: (2286,)


In [9]:
print("Local train days:")
print(train.loc[local_train_mask, "day"].value_counts().sort_index())

print("\nLocal validation days:")
print(train.loc[local_valid_mask, "day"].value_counts().sort_index())

print("\nLocal train datetime range:")
print(train.loc[local_train_mask, "datetime"].min())
print(train.loc[local_train_mask, "datetime"].max())

print("\nLocal validation datetime range:")
print(train.loc[local_valid_mask, "datetime"].min())
print(train.loc[local_valid_mask, "datetime"].max())

Local train days:
day
1     575
2     573
3     573
4     574
5     575
6     572
7     574
8     574
9     575
10    572
11    568
12    573
13    574
14    574
15    574
Name: count, dtype: int64

Local validation days:
day
16    574
17    575
18    563
19    574
Name: count, dtype: int64

Local train datetime range:
2011-01-01 00:00:00
2012-12-15 23:00:00

Local validation datetime range:
2011-01-16 00:00:00
2012-12-19 23:00:00


## Stage 4 setup notes

This notebook performs controlled model comparison for time-aware regression.

Data used:

- only `train.csv`

Official Kaggle `test.csv` is not used for local validation because it has no `count`.

Target:

- `count`

Main feature set excludes:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day`

Leakage control:

- `casual` and `registered` are excluded because they are target components.
- raw `datetime` is excluded after datetime-derived feature extraction.
- `day` is excluded from the main comparison because it may transfer poorly to official test days 20–31.

Local validation split:

- local train: days 1–15
- local validation: days 16–19

No random split is used.
No official test data is used.
No hyperparameter tuning is performed at this stage.
No log-target transformation is used in the main comparison.

## Controlled model comparison

In [15]:
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error

In [16]:
def clipped_predictions(y_pred):
    return np.clip(y_pred, a_min=0, a_max=None)


def regression_metrics(y_true, y_pred):
    y_pred_clipped = clipped_predictions(y_pred)

    rmsle = np.sqrt(mean_squared_log_error(y_true, y_pred_clipped))
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_clipped))
    mae = mean_absolute_error(y_true, y_pred_clipped)

    return {
        "RMSLE": rmsle,
        "RMSE": rmse,
        "MAE": mae,
        "min_pred_raw": np.min(y_pred),
        "min_pred_clipped": np.min(y_pred_clipped),
        "max_pred_raw": np.max(y_pred),
    }

In [17]:
models = {
    "DummyRegressor_mean": DummyRegressor(strategy="mean"),
    "LinearRegression": LinearRegression(),
    "Ridge": Ridge(),
    "DecisionTree_depth3": DecisionTreeRegressor(max_depth=3, random_state=42),
    "RandomForestRegressor": RandomForestRegressor(random_state=42, n_jobs=-1),
    "ExtraTreesRegressor": ExtraTreesRegressor(random_state=42, n_jobs=-1),
    "GradientBoostingRegressor": GradientBoostingRegressor(random_state=42),
    "HistGradientBoostingRegressor": HistGradientBoostingRegressor(random_state=42),
}

In [18]:
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_valid)

    metrics = regression_metrics(y_valid, y_pred)
    metrics["model"] = model_name
    results.append(metrics)

results_df = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values("RMSLE")
)

results_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw
model,,,,,,
RandomForestRegressor,0.352386,52.494191,30.632303,1.760000,1.760000,926.280000
ExtraTreesRegressor,0.360769,51.676713,30.595973,1.130000,1.130000,896.530000
HistGradientBoostingRegressor,0.451918,45.782780,27.786296,-17.168972,0.000000,905.601978
GradientBoostingRegressor,0.737834,70.252615,46.616748,-73.913768,0.000000,699.438491
DecisionTree_depth3,0.767086,130.765166,91.359668,15.143469,15.143469,375.426947
Ridge,1.242683,146.371947,106.992302,-123.156212,0.000000,438.767285
LinearRegression,1.242720,146.371005,106.992386,-123.176376,0.000000,438.779023
DummyRegressor_mean,1.531283,183.076977,142.644424,190.530233,190.530233,190.530233


In [19]:
stage3_baseline_rmsle = 0.767086

results_df["RMSLE_delta_vs_stage3_tree"] = (
    results_df["RMSLE"] - stage3_baseline_rmsle
)

results_df

,RMSLE,RMSE,MAE,min_pred_raw,min_pred_clipped,max_pred_raw,RMSLE_delta_vs_stage3_tree
model,,,,,,,
RandomForestRegressor,0.352386,52.494191,30.632303,1.760000,1.760000,926.280000,-4.146995e-01
ExtraTreesRegressor,0.360769,51.676713,30.595973,1.130000,1.130000,896.530000,-4.063174e-01
HistGradientBoostingRegressor,0.451918,45.782780,27.786296,-17.168972,0.000000,905.601978,-3.151678e-01
GradientBoostingRegressor,0.737834,70.252615,46.616748,-73.913768,0.000000,699.438491,-2.925156e-02
DecisionTree_depth3,0.767086,130.765166,91.359668,15.143469,15.143469,375.426947,-2.815939e-07
Ridge,1.242683,146.371947,106.992302,-123.156212,0.000000,438.767285,4.755975e-01
LinearRegression,1.242720,146.371005,106.992386,-123.176376,0.000000,438.779023,4.756342e-01
DummyRegressor_mean,1.531283,183.076977,142.644424,190.530233,190.530233,190.530233,7.641971e-01


### Controlled model comparison findings

- All models were trained on the same local train split: days 1–15.
- All models were evaluated on the same local validation split: days 16–19.
- Official Kaggle `test.csv` was not used.
- No random split was used.
- No hyperparameter tuning was performed.
- No log-target transformation was used in the main comparison.
- `casual` and `registered` were not used.

Best model by RMSLE:

- `<fill>`

Comparison against Stage 3 shallow tree baseline:

- Stage 3 `DecisionTreeRegressor(max_depth=3)` RMSLE: 0.767086
- Best Stage 4 model RMSLE: `<fill>`
- Improvement: `<fill>`

Candidate models for later tuning/log-target experiments:

- `<fill>`
- `<fill>`
- `<fill>`

## Stage 4 conclusions

### Controlled setup

All models used the same controlled setup:

- data source: `train.csv` only
- target: `count`
- local train: days 1–15
- local validation: days 16–19
- official Kaggle `test.csv` was not used
- random split was not used
- no hyperparameter tuning was performed
- no log-target transformation was used in the main comparison

### Feature set

Main feature set:

- `season`
- `holiday`
- `workingday`
- `weather`
- `temp`
- `atemp`
- `humidity`
- `windspeed`
- `year`
- `month`
- `hour`
- `weekday`
- `is_weekend`

Excluded from `X`:

- `count`
- `casual`
- `registered`
- raw `datetime`
- `day`

`casual` and `registered` were excluded because they are target components.

### Models compared

- `DummyRegressor(strategy="mean")`
- `LinearRegression`
- `Ridge`
- `DecisionTreeRegressor(max_depth=3, random_state=42)`
- `RandomForestRegressor(random_state=42, n_jobs=-1)`
- `ExtraTreesRegressor(random_state=42, n_jobs=-1)`
- `GradientBoostingRegressor(random_state=42)`
- `HistGradientBoostingRegressor(random_state=42)`

### Metrics table

| model | RMSLE | RMSE | MAE |
|---|---:|---:|---:|
| RandomForestRegressor | 0.352386 | 52.494191 | 30.632303 |
| ExtraTreesRegressor | 0.360769 | 51.676713 | 30.595973 |
| HistGradientBoostingRegressor | 0.451918 | 45.782780 | 27.786296 |
| GradientBoostingRegressor | 0.737834 | 70.252615 | 46.616748 |
| DecisionTree_depth3 | 0.767086 | 130.765166 | 91.359668 |
| Ridge | 1.242683 | 146.371947 | 106.992302 |
| LinearRegression | 1.242720 | 146.371005 | 106.992386 |
| DummyRegressor_mean | 1.531283 | 183.076977 | 142.644424 |

### Best model by primary metric

Primary metric: RMSLE.

Best model:

- `RandomForestRegressor`

Validation metrics:

- RMSLE: 0.352386
- RMSE: 52.494191
- MAE: 30.632303

### Comparison against Stage 3 baseline

Stage 3 best simple baseline:

- `DecisionTreeRegressor(max_depth=3, random_state=42)`
- RMSLE: 0.767086

Stage 4 best model:

- `RandomForestRegressor`
- RMSLE: 0.352386

Improvement:

- RMSLE delta: -0.414700

This is a large improvement over the shallow tree baseline.

### Interpretation

Tree ensembles strongly improve over the shallow decision tree and linear models.

This is expected because bike demand has nonlinear structure:

- strong hourly patterns;
- different hourly profiles for working and non-working days;
- seasonal effects;
- weather effects;
- interactions between calendar/time/weather variables.

However, conclusions are based only on the fixed local validation split and still need later validation through the next approved stages.

### Candidate models for later tuning / log-target experiments

Recommended candidates:

1. `RandomForestRegressor`
   - best RMSLE in this comparison.

2. `ExtraTreesRegressor`
   - close to RandomForest by RMSLE and slightly better RMSE/MAE.

3. `HistGradientBoostingRegressor`
   - best RMSE and MAE, but weaker RMSLE than RandomForest/ExtraTrees.

Optional candidate:

- `GradientBoostingRegressor`
  - improved only slightly over the shallow tree by RMSLE in this default configuration.

### Prediction clipping

Predictions were clipped at 0 before RMSLE calculation.

Negative raw predictions were observed for:

- `HistGradientBoostingRegressor`
- `GradientBoostingRegressor`
- `Ridge`
- `LinearRegression`

This is important to monitor because negative bike rental demand is not meaningful.

### Leakage checks

Passed:

- `casual` not in `X`
- `registered` not in `X`
- `count` not in `X`
- raw `datetime` not in `X`
- `day` excluded from the main comparison
- official Kaggle `test.csv` not used for validation
- random split not used
- no hyperparameter tuning
- no log-target transformation in the main comparison